# 2_4_02 - Silver custom_activites_processed_fields

Nightly child of 2_4_01: explodes the `fields` array<struct> from the rebuilt
`custom_activites_processed` parent into one row per field definition per activity
type (id, name, type, required, description...). This is the dictionary that maps
`cf_...` custom-field keys to human names - gold (3_03 sales_fact) resolves field
ids through this table instead of hardcoding them.

Parent-driven delete+append CDC, mirroring 2_1_03_01 (contacts child task).
NOT nightly-maintained by design: the two permission children
(`editable_with_roles`, `fields_editable_with_roles`) - no analytics consumer;
they hold the one-off materialization from `rebuild_silver_children_from_parents`.

In [0]:
%run ../utils

In [0]:
# Create custom_activites_processed_fields from the SILVER parent custom_activites_processed
# (the parent is built first by 2_4_01 - children can never be newer than the parent).
# Mirrors 2_1_03_01: parent-driven delete+append CDC via explode_to_child.
# recurse=False: writes ONLY the fields table. The fields_editable_with_roles grandchild
# is NOT maintained nightly (permission metadata, no consumer) - it exists from the
# Nov 9 2026 child rebuild only.
import time
from pyspark.sql.functions import col, lit
start = time.time()

print(f"{'='*80}")
print(f"custom_activites_processed -> custom_activites_processed_fields")
print(f"{'='*80}")

# Check CDC watermark for the fields child
watermark = get_silver_watermark('custom_activites_processed_fields')
if watermark:
    print(f"  CDC MODE: reading parent changes after {watermark}")
else:
    print(f"  FULL LOAD: first run")

# Step 1: Read the silver parent
PARENT_TABLE = 'crm_ingestion.silver.custom_activites_processed'
CHILD_TABLE = 'custom_activites_processed_fields'
df_flat = spark.table(PARENT_TABLE)
if watermark:
    df_flat = df_flat.filter(col("bronze_updated_at") > lit(watermark))

n_changed = df_flat.count()
if n_changed == 0:
    print(f"  CDC: No new parent rows since {watermark}")
    all_tables = spark.catalog.listTables("crm_ingestion.silver")
    related = [t for t in all_tables if t.name == CHILD_TABLE]
    print(f"\n  Existing silver tables ({len(related)}):")
    for t in related:
        tdf = spark.table(f"crm_ingestion.silver.{t.name}")
        cnt = tdf.count()
        cols = tdf.columns
        print(f"     {t.name} ({len(cols)} cols, {cnt:,} rows): {cols}")
    print(f"{'='*80}")
else:
    print(f"  Source: {n_changed:,} parent rows")

    # Step 2: Find PK and build carry columns (parent's own PK - pinned to id
    # by the parent rebuild; find_pk confirms it on the changed slice)
    pk = find_pk(df_flat)
    carry = list(dict.fromkeys(["bronze_insert_date", "bronze_updated_at"] + ([pk] if pk else [])))

    # Step 3: Explode the fields array with dry_run=False to actually write the table.
    # CDC FIX (same as 2_1_03_01): propagate the watermark to write_silver (module
    # global via %run) so the write is delete+append matching the filtered read.
    _CDC_WATERMARK = watermark
    if "fields" in df_flat.columns:
        print(f"\n  Exploding fields array -> {CHILD_TABLE} (recurse=False - no grandchild):")
        explode_to_child(df_flat, 'custom_activites_processed', 'fields', carry,
                         depth=0, num_path="1_2", parent_pk=pk, dry_run=False,
                         recurse=False)
        print(f"\n  LINEAGE TREE:")
        for num, depth, name in _lineage_tree:
            print(f"  # * {'---' * depth}{num}_{name}")
    else:
        print("  WARNING: No fields array found")

    # Reset CDC watermark (mirrors process_table hygiene)
    _CDC_WATERMARK = None

    print(f"\n  Completed in {time.time() - start:.1f}s")
    print(f"{'='*80}")

In [0]:
display(spark.table('crm_ingestion.silver.custom_activites_processed_fields'))